In [77]:
import os
import numpy as np
from pathlib import Path
import json
from sklearn.decomposition import PCA

In [56]:


path =  Path("/media/simon/42A099D63E90C520/CVAT_Data/Booster_K1_Log_02-26")
annot_json = path / "annotations/default.json"
img_path = path / "images/default"
with open(annot_json, "r") as f:
    data = json.load(f)

In [36]:
id_to_name = {i: n["name"] for i, n in enumerate(data["categories"]["label"]["labels"])}
name_to_id = {n: i for i, n in id_to_name.items()}

In [88]:
def _normalize_coords(coords, img_sz):
    return [c/img_sz[(i+1)%2] for i,c in enumerate(coords)]
    # coords[1::2] = [c/img_sz[0] for c in coords[1::2]]
    # return coords

def _load_annots_for_image(img_data):
    for a in img_data["annotations"]:
        a_id = a["label_id"]
        a_name = id_to_name[a_id]
        a_type = a["type"]
        if a_type == 'bbox':
            content = a["bbox"]
        elif a_type in ['polygon']:
            content = a["points"]
            img_sz = img["image"]["size"]
            # return [[content[i], content[i+1] for i in range(len(content))]
            normed = _normalize_coords(content, img_sz)
            return [normed[::2], normed[1::2]]
        # return content

In [91]:
def _rotate_points_by_angle(points, angle, center=None):
    c, s = np.cos(angle), np.sin(angle)
    rotation_matrix = np.array(((c, -s), (s, c)))
    if center is None:
        return np.dot(points, rotation_matrix.T)
    else:
        return np.dot(points-center, rotation_matrix.T) + center

# for img in data["items"]:
#     img_path = path / img["image"]["path"]
#     annots = _load_annots_for_image(img)
points = np.array(_load_annots_for_image(data["items"][0])).T
# points = np.stack(raw_points[::2], raw_points[1::2])
center = points.mean(axis=0)
centered_points = points - center
pca = PCA(n_components=2)
pca.fit(centered_points)
hauptachse_richtung = pca.components_[0] # Erste Hauptachse (längste Achse)
nebenachse_richtung = pca.components_[1] # Zweite Hauptachse
angle = np.degrees(np.arctan2(*hauptachse_richtung)) % 360 # deliberately filled y, x params "incorrectly".

# Rotate points around their center to align hauptachse
rotated_points = _rotate_points_by_angle(centered_points, angle)
shifted_points = rotated_points + center

minx, maxx = min(shifted_points[:, 0]), max(shifted_points[:, 0])
miny, maxy = min(shifted_points[:, 1]), max(shifted_points[:, 1])

bb = np.array([[minx, miny], [maxx, miny], [maxx, maxy], [minx, maxy]])
obb = _rotate_points_by_angle(bb, -angle, center)
obb = np.clip(obb, 0, 1)

print(f"Schwerpunkt: {center}")
# print(f"Richtung der Hauptachse: {hauptachse_richtung}")
print(f"Rotation: {angle}")
print(f"OBB: {obb}")

Schwerpunkt: [0.44439338 0.10117188]
Rotation: 3.5350306641334694
OBB: [[0.48710437 0.14377328]
 [0.44491498 0.16128523]
 [0.4021508  0.05825875]
 [0.44434019 0.0407468 ]]
